In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
base_path = "/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images"

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Training — augmentation + MobileNetV2 preprocessing
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,   
    validation_split=0.2,
    rotation_range=180,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)
train_generator = train_datagen.flow_from_directory(
    base_path,
    target_size=(224, 224),                    
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

# Validation — preprocessing only, NO augmentation
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,   # ← fixed
    validation_split=0.2
)
val_generator = val_datagen.flow_from_directory(
    base_path,
    target_size=(224, 224),                    # ← fixed
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print(train_generator.class_indices)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

callbacks = [
    ModelCheckpoint("mobilenet_malaria.keras", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    CSVLogger("mobilenetv2_malaria_log.csv")
]

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
base_model = MobileNetV2(         
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)      
)

# Freeze base
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(train_generator.num_classes, activation="softmax")(x)

mobilenet_model = Model(inputs=base_model.input, outputs=outputs) 

In [ ]:
mobilenet_model.summary()

In [ ]:
mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"Total parameters : {mobilenet_model.count_params():,}")
print(f"Output classes   : {train_generator.num_classes}")

In [ ]:
history = mobilenet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
mobilenet_model.save("mobilenet_malaria.keras")

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

val_generator.reset()   # ← reset internal state

preds = mobilenet_model.predict(
    val_generator,
    steps=len(val_generator),   # ← ceil(5510 / 32) = 173 steps, no wrap-around
    verbose=1
)

pred_labels = np.argmax(preds, axis=1)
true_labels = val_generator.classes   # ground truth in order

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix:")
print(cm)

In [ ]:
import seaborn as sns

plt.figure(figsize=(5,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Parasitized','Uninfected'],
            yticklabels=['Parasitized','Uninfected'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(true_labels, pred_labels, target_names=['Parasitized','Uninfected']))